# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I choose a random forest ensemble because it can capture nonlinear interactions among page metrics, handle mixed numeric and categorical features, and still produce an interpretable ranking of important signals. I include logistic regression and a decision tree as simpler comparison models to benchmark how much complexity is worth in this lane.

In [12]:
# Goal: Choose a modeling approach and explain why it fits this lane.
method_choice = "Random forest with logistic regression and decision tree as comparison models"
method_reason = (
    "Random forest is robust to nonlinear interactions among page metrics, "
    "handles mixed numeric and categorical signals, and is a good practical choice for refresh recommendations. "
    "Decision tree and logistic regression provide simpler baselines for interpretation and comparison."
)
method_choice, method_reason

('Random forest with logistic regression and decision tree as comparison models',
 'Random forest is robust to nonlinear interactions among page metrics, handles mixed numeric and categorical signals, and is a good practical choice for refresh recommendations. Decision tree and logistic regression provide simpler baselines for interpretation and comparison.')

## 2. Split design

I use a client-aware holdout split because the problem is aligned with making refresh decisions across different clients. This prevents leakages from the same client appearing in both train and test sets, which is more honest than a row-level split for this lane.

In [13]:
# Goal: Define an honest split strategy for model training.
# We use a client-aware holdout split, matching the Week-4 baseline approach.
import sys
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

repo_root = Path.cwd()
while repo_root.name != "flyrank-internship" and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "scripts"))

from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, normalize, percentile_rank
import pandas as pd

feature_path = repo_root / "data" / "processed" / "refresh_feature_vector.csv"
if not feature_path.exists():
    raise FileNotFoundError(f"Missing processed data at {feature_path}")

frame = pd.read_csv(feature_path)

# Build features + labels to match training pipeline
feature_columns = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
frame = frame.dropna(subset=["is_declining_label"])

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(frame, groups=frame["client_id"]))
train = frame.iloc[train_idx].reset_index(drop=True)
test = frame.iloc[test_idx].reset_index(drop=True)

split_design = {
    "strategy": "client-holdout",
    "train_clients": int(train["client_id"].nunique()),
    "test_clients": int(test["client_id"].nunique()),
    "train_rows": len(train),
    "test_rows": len(test),
}
split_design

{'strategy': 'client-holdout',
 'train_clients': 25,
 'test_clients': 7,
 'train_rows': 23837,
 'test_rows': 6163}

## 3. Train + compare vs my baseline

I train on the same feature set and client-aware split as the Week-4 baseline, then compare precision and recall. This keeps the comparison meaningful and honest.

In [14]:
# Goal: Train models and compare them to the Week-4 baseline on the same split.
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score

numeric_features = [col for col in MODEL_NUMERIC_FEATURES if col in frame.columns]
categorical_features = [col for col in MODEL_CATEGORICAL_FEATURES if col in frame.columns]
feature_cols = numeric_features + categorical_features

X_train = train[feature_cols].copy()
X_test = test[feature_cols].copy()

y_train = train["is_declining_label"].astype(int)
y_test = test["is_declining_label"].astype(int)

X_train_numeric = X_train[numeric_features].apply(pd.to_numeric, errors="coerce")
X_test_numeric = X_test[numeric_features].apply(pd.to_numeric, errors="coerce")

X_train_numeric = X_train_numeric.replace([pd.NA], 0).fillna(0)
X_test_numeric = X_test_numeric.replace([pd.NA], 0).fillna(0)

X_train_categorical = X_train[categorical_features].fillna("unknown").astype(str)
X_test_categorical = X_test[categorical_features].fillna("unknown").astype(str)

X_train_cat = pd.get_dummies(
    X_train_categorical,
    prefix=categorical_features,
    dummy_na=False,
    dtype=float,
)
X_test_cat = pd.get_dummies(
    X_test_categorical,
    prefix=categorical_features,
    dummy_na=False,
    dtype=float,
)

X_train = pd.concat([X_train_numeric.reset_index(drop=True), X_train_cat.reset_index(drop=True)], axis=1)
X_test = pd.concat([X_test_numeric.reset_index(drop=True), X_test_cat.reset_index(drop=True)], axis=1)

# Align one-hot encoded columns between train and test, filling missing dummy columns with 0.
X_train, X_test = X_train.align(X_test, join="outer", axis=1, fill_value=0)

models = {
    "logistic_regression": LogisticRegression(max_iter=1000, random_state=42),
    "decision_tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=100, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_test)
    results.append({
        "model": name,
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probs),
    })

# Reconstruct the baseline score on the same test split, since the processed baseline file may not exist.
baseline_test = test.copy()
baseline_test["visibility_score"] = percentile_rank(np.log1p(baseline_test["impressions_90d"]))
baseline_test["freshness_risk_score"] = percentile_rank(baseline_test["days_since_last_update"])
baseline_test["position_opportunity_score"] = (
    (1 - normalize(baseline_test["avg_position"].clip(lower=1, upper=50)))
    * baseline_test["visibility_score"]
    * (baseline_test["avg_position"] > 0).astype(int)
)
baseline_test["depth_gap_score"] = (1 - percentile_rank(baseline_test["word_count"])) * baseline_test["visibility_score"]
baseline_test["baseline_refresh_score"] = (
    0.40 * baseline_test["visibility_score"]
    + 0.30 * baseline_test["freshness_risk_score"]
    + 0.25 * baseline_test["position_opportunity_score"]
    + 0.05 * baseline_test["depth_gap_score"]
).clip(0, 1)

baseline_scores = baseline_test["baseline_refresh_score"].astype(float)
baseline_preds = (baseline_scores >= 0.5).astype(int)
baseline_precision = precision_score(y_test, baseline_preds, zero_division=0)
baseline_recall = recall_score(y_test, baseline_preds, zero_division=0)

results.append({
    "model": "baseline_score",
    "precision": baseline_precision,
    "recall": baseline_recall,
    "roc_auc": None,
})

import pandas as pd
model_comparison = pd.DataFrame(results)
model_comparison

/home/otto/Documents/projects/flyrank-internship/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,model,precision,recall,roc_auc
0,logistic_regression,0.572324,0.696094,0.611231
1,decision_tree,0.593699,0.574468,0.594308
2,random_forest,0.583183,0.614481,0.609241
3,baseline_score,0.474788,0.337885,NaN


## 4. Errors and interpretation

I analyze false positives and false negatives from the random forest, and I surface the top features that drive its decisions. This helps understand where the model may still be wrong and what signals it leans on.

In [15]:
# Goal: Analyze where the model still makes mistakes and which features matter.
model_feat_names = X_train.columns.tolist()
feature_importances = pd.DataFrame({
    "feature": model_feat_names,
    "importance": models["random_forest"].feature_importances_,
}).sort_values("importance", ascending=False).head(10)

preds = models["random_forest"].predict(X_test)
errors = test.copy()
errors["predicted"] = preds
errors["false_positive"] = (errors["predicted"] == 1) & (errors["is_declining_label"] == 0)
errors["false_negative"] = (errors["predicted"] == 0) & (errors["is_declining_label"] == 1)

page_columns = [col for col in ["page_type", "content_type", "client_id"] if col in errors.columns]
summary = {
    "false_positive_rate": errors["false_positive"].mean(),
    "false_negative_rate": errors["false_negative"].mean(),
}
if "page_type" in errors.columns:
    summary["top_false_positive_pages"] = (
        errors[errors["false_positive"]]["page_type"].value_counts().head(5).to_dict()
    )
    summary["top_false_negative_pages"] = (
        errors[errors["false_negative"]]["page_type"].value_counts().head(5).to_dict()
    )
else:
    summary["top_false_positive_types"] = errors[errors["false_positive"]][page_columns].head(5).to_dict(orient="records")
    summary["top_false_negative_types"] = errors[errors["false_negative"]][page_columns].head(5).to_dict(orient="records")

feature_importances, summary

(                  feature  importance
 32    log_impressions_90d    0.102926
 5            avg_position    0.097595
 19  days_with_impressions    0.088199
 12       content_age_days    0.070065
 46             word_count    0.058778
 6              char_count    0.057285
 33       log_sessions_90d    0.054039
 20     days_with_sessions    0.050945
 44            scroll_rate    0.050025
 17                    ctr    0.049342,
 {'false_positive_rate': np.float64(0.2244036994969982),
  'false_negative_rate': np.float64(0.19698198929092975),
  'top_false_positive_types': [{'content_type': 'keyword article',
    'client_id': 'client_8527a891e2'},
   {'content_type': 'keyword article', 'client_id': 'client_4e07408562'},
   {'content_type': 'keyword article', 'client_id': 'client_f369cb89fc'},
   {'content_type': 'keyword article', 'client_id': 'client_f369cb89fc'},
   {'content_type': 'keyword article', 'client_id': 'client_8527a891e2'}],
  'top_false_negative_types': [{'content_type': 'key

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.